In [1]:
import sys
print(sys.executable)

D:\conda_envs\spark_local\python.exe


In [5]:
path_bronze = r"D:\data-ai-engineer-roadmap\nyc_taxi\data\bronze\raw_files\yellow\2025\yellow_tripdata_2025-05.parquet"
env = "local"
periode = 202505
taxi_type = "yellow"

In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [3]:
from nyc_taxi.src.common.pipeline_context import PipelineContext
from nyc_taxi.src.common.decorators import log_execution 
from nyc_taxi.src.common.pipeline_step import PipelineStep
from nyc_taxi.src.exception.exception_handler import DataNotAvailableError
from nyc_taxi.src.bronze.uber_bronze import UberBronze
from nyc_taxi.src.silver.uber_silver import UberSilver
from nyc_taxi.src.logger.logger import PipelineLogger
from nyc_taxi.src.spark.spark_manager import SparkManager

In [6]:
logger = PipelineLogger(
        name="pipeline_runner",
        env=env,
        periode=periode,
        taxi_type=taxi_type
)

2026-10-27 18:23:49 | INFO | ====================================================================================================
2026-10-27 18:23:49 | INFO | Informations de démarrage
2026-10-27 18:23:49 | INFO | ====================================================================================================
2026-10-27 18:23:49 | INFO | PIPELINE LOGGER
2026-10-27 18:23:49 | INFO | environment      = local
2026-10-27 18:23:49 | INFO | periode          = 202505
2026-10-27 18:23:49 | INFO | taxi_type        = yellow
2026-10-27 18:23:49 | INFO | path_logs        = D:/data-ai-engineer-roadmap/nyc_taxi/logs
2026-10-27 18:23:49 | INFO | log_file         = D:\data-ai-engineer-roadmap\nyc_taxi\logs\yellow\2025\202505_20261027_182349.log
2026-10-27 18:23:49 | INFO | log_sql_file     = D:\data-ai-engineer-roadmap\nyc_taxi\logs\yellow\2025\202505_20261027_182349.sql
2026-10-27 18:23:49 | INFO | logging_mode     = console + file
2026-10-27 18:23:49 | INFO | ====================================

In [7]:
spark_manager = SparkManager(
    app_name="nyc_taxi_pipeline",
    env=env,
    logger=logger
)

2026-10-27 18:23:55 | INFO | project_root : D:\data-ai-engineer-roadmap\nyc_taxi
2026-10-27 18:23:55 | INFO | Loading config from D:\data-ai-engineer-roadmap\nyc_taxi\config\variable_environnement.yaml


In [8]:
spark = spark_manager.get_spark()

2026-10-27 18:24:09 | INFO | warehouse = file:/D:/data-ai-engineer-roadmap/nyc_taxi/spark-warehouse
2026-10-27 18:24:09 | INFO | metastore = jdbc:derby:D:/data-ai-engineer-roadmap/nyc_taxi/metastore_db;create=true
2026-10-27 18:24:09 | INFO | warehouse_yaml = D:/data-ai-engineer-roadmap/nyc_taxi/spark-warehouse
2026-10-27 18:24:09 | INFO | metastore_yaml = D:/data-ai-engineer-roadmap/nyc_taxi/metastore_db
2026-10-27 18:24:09 | INFO | Spark local dir = D:/spark_tmp
2026-10-27 18:24:09 | INFO | ========================================================================================================================
2026-10-27 18:24:09 | INFO | SPARK ENVIRONMENT
2026-10-27 18:24:09 | INFO | ========================================================================================================================
2026-10-27 18:24:09 | INFO | environment          = local
2026-10-27 18:24:09 | INFO | user                 = alpha
2026-10-27 18:24:09 | INFO | hostname             = DIOP
2026-10-27 

In [10]:
df_rejected = (
    spark.table("audit.silver_rejected_rows")
    #.where(f"Periode = {periode}")
)

AnalysisException: org.apache.hadoop.hive.ql.metadata.HiveException: java.lang.RuntimeException: Unable to instantiate org.apache.hadoop.hive.ql.metadata.SessionHiveMetaStoreClient

In [ ]:
df_silver = (
    spark.table("silver.silver_nyc_taxi")
    .where(f"Periode = {periode}")
)

In [ ]:
from pyspark.sql import functions as F

colonnes_numeriques = [
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "passenger_count",
    "trip_duration_minutes",
    "average_speed"
]

colonnes_disponibles = [
    c for c in colonnes_numeriques
    if c in df_silver.columns
]

df_stats = df_silver.agg(
    F.count("*").alias("nb_lignes"),
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(f"{c}_null")
        for c in colonnes_disponibles
    ],
    *[
        F.min(c).alias(f"{c}_min")
        for c in colonnes_disponibles
    ],
    *[
        F.max(c).alias(f"{c}_max")
        for c in colonnes_disponibles
    ],
    *[
        F.round(F.avg(c), 2).alias(f"{c}_moyenne")
        for c in colonnes_disponibles
    ]
)

display(df_stats)